# Use the AppEEARS API to get NDVI data for your location of interest

Set up: To get started on this notebook, you’ll need to restore any variables from previous notebooks to your workspace. To save time and memory, make sure to specify which variables you want to load.

In [1]:
%store -r

c:\Users\nyca4176\AppData\Local\miniconda3\envs\earth-analytics-python-2026\Lib\site-packages\pyproj\network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


You will also need to import any libraries you are using in this
notebook, since they won’t carry over from the previous notebook:

In [2]:
### import libraries

### file structure
import os
import pathlib

### plotting
import hvplot.pandas
import hvplot.xarray

### dealing with spatial data
import geopandas as gpd
import rioxarray as rxr
import xarray as xr
import pandas as pd

### earthpy
import earthpy
import earthpy.api.appeears as eaapp

###  additional packages
### Helps with Zip files
import zipfile

### Helps with API keys
import requests 

### quick plots
import matplotlib.pyplot as plt

In [3]:
import os, pyproj
print("env PROJ_LIB:", os.environ.get("PROJ_LIB"))
print("env PROJ_DATA:", os.environ.get("PROJ_DATA"))
print("pyproj data dir:", pyproj.datadir.get_data_dir())

env PROJ_LIB: None
env PROJ_DATA: C:\Users\nyca4176\AppData\Local\miniconda3\envs\earth-analytics-python-2026\Library\share\proj
pyproj data dir: C:\Users\nyca4176\AppData\Local\miniconda3\envs\earth-analytics-python-2026\Library\share\proj


In [4]:
%store -r

## File Management 

In [5]:
### create directory for data
boundary_dir = os.path.join(
    portfolio_dir, "coding-assignments", "03_vegetation", "data", "06_boundary"
)

ndvi_dir = os.path.join(
    portfolio_dir, "coding-assignments", "03_vegetation", "data", "06_ndvi"
)

### make the folders
os.makedirs(boundary_dir, exist_ok=True)
os.makedirs(ndvi_dir, exist_ok=True)

### make folders paths
boundary_dir = pathlib.Path(boundary_dir)
ndvi_dir = pathlib.Path(ndvi_dir)

boundary_dir, ndvi_dir

(WindowsPath('C:/Users/nyca4176/Repos/earth-data-portfolio/coding-assignments/03_vegetation/data/06_boundary'),
 WindowsPath('C:/Users/nyca4176/Repos/earth-data-portfolio/coding-assignments/03_vegetation/data/06_ndvi'))

In [6]:
### home for the zip file (from tutorial)
zip_nps_path = boundary_dir / "nps_boundaries.gdb.zip"
zip_nps_path

WindowsPath('C:/Users/nyca4176/Repos/earth-data-portfolio/coding-assignments/03_vegetation/data/06_boundary/nps_boundaries.gdb.zip')

## Get extent of interest
prepare the shape file used to get NDVI data


In [7]:
### specify URL for api access to the data
nps_url = (
    "https://irma.nps.gov/DataStore/DownloadFile/765006"
    "?Reference=231443"
)

In [11]:
### download the zip file from the tutorial
if not zip_nps_path.exists():

    ### make a temp file name
    temp_path = zip_nps_path.with_suffix(".part")

    ### download data
    with requests.get(nps_url,
                      stream = True,
                      timeout = (30,300)) as response:
            
            response.raise_for_status()

            ### open temp file
            with temp_path.open("wb") as f:
                
                ### read in 1 chunk at a time
                for chunk in response.iter_content(chunk_size = 1024*1024):  ## 1MB
                    
                    ### skip writing empty chunks
                    if chunk:
                        
                        ## write chunk to temp file
                        f.write(chunk)

            ### rename temp file to final file
            temp_path.rename(zip_nps_path)

In [12]:
### extract zipped data

if not list (boundary_dir.rglob("*.gdb")):

    ### create directory to extract the zip file to
    ### this keeps the folder name from zip metadata as a subfolder of the boundary_dir
    boundary_dir.mkdir(parents=True, exist_ok=True)

    ### open zip folder
    with zipfile.ZipFile(zip_nps_path, "r") as archive:
        archive.extractall(boundary_dir)


In [13]:
### find the NPS geodatabase
gdb_paths = sorted(boundary_dir.rglob("*.gdb"))

### check there is only one
gdb_paths

### select first gdb
gdb_path = gdb_paths[0]

In [14]:
### check out layers in geodatabase
layers_nps = gpd.list_layers(gdb_path)

layers_nps

c:\Users\nyca4176\AppData\Local\miniconda3\envs\earth-analytics-python-2026\Lib\site-packages\pyogrio\core.py:39: RuntimeWarning: Could not detect PROJ data files. Set PROJ_LIB environment variable to the correct path.
  _init_proj_data()


,name,geometry_type
0,NPS_Boundaries,MultiPolygon


In [15]:
### find actual park boundary polygons

polygons_nps = layers_nps [
    ### filter to geon cols
    layers_nps["geometry_type"]

    ### replace missing values with empty strings
    .fillna("")

    ### extract polygons
    .str.contains("polygon", case = False)

]

polygons_nps

,name,geometry_type
0,NPS_Boundaries,MultiPolygon


In [16]:
### boundaries only
boundary_layers = polygons_nps[
    polygons_nps["name"].str.contains("bound", case=False)
]
boundary_layers

,name,geometry_type
0,NPS_Boundaries,MultiPolygon


In [17]:
### define layer name
layer_name = boundary_layers.iloc[0]["name"]

In [ ]:
### open data
nps_units = gpd.read_file(gdb_path, layer = layer_name)

### getting package error CRSError: Invalid projection: EPSG:6318: (Internal Proj Error: proj_create: no database context specified)


c:\Users\nyca4176\AppData\Local\miniconda3\envs\earth-analytics-python-2026\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined. Further messages of this type will be suppressed.
  return ogr_read(
c:\Users\nyca4176\AppData\Local\miniconda3\envs\earth-analytics-python-2026\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts.  The processing may be really slow.  You can skip the processing by setting METHOD=SKIP. Further messages of this type will be suppressed.
  return ogr_read(


CRSError: Invalid projection: EPSG:6318: (Internal Proj Error: proj_create: no database context specified)

In [19]:
### extract rocky mountain np
romo_gdf = nps_units.loc[nps_units["unit_name"] == "Rocky Mountain National Park"]

NameError: name 'nps_units' is not defined

In [20]:
### plot the gdf
romo_gdf.plot (figsize = (10, 6), edgecolor = "black", facecolor = "none")

NameError: name 'romo_gdf' is not defined

## Exploring the AppEEARS API for NASA Earthdata access

Before you get started with the data download, you will need a free [NASA Earthdata account](https://urs.earthdata.nasa.gov/home) if you don’t have one already.

You will download MODIS NDVI data for the study period. MODIS is a multispectral instrument that measures red and NIR data (and so can be used for NDVI). There are two MODIS sensors on two different platforms: satellites Terra and Aqua.

You can learn more about MODIS datasets and the science they support [here](https://modis.gsfc.nasa.gov/).

Since we’re asking for a special download that only covers our study area, we can’t just find a link to the data. Instead, we have to negotiate with the data server. We’re doing this using the
[APPEEARS](https://appeears.earthdatacloud.nasa.gov/api/) API (Application Programming Interface). The API makes it possible for you to request data using code. You can use code from the `earthpy` library to handle the API request.

Often when we want to do something more complex in coding, we find an example and modify it. This download code is already almost a working example. Your task will be to:   

- Replace the start and end dates in the task parameters. Download data from July, when greenery is at its peak in the Northern Hemisphere.  
- Replace the year range. You should get two 3 year periods so you can see the change.  
- Replace `gdf` with the name of your site geodataframe.  
- Enter your NASA Earthdata username and password when prompted at the top of your screen.


**Note:** It can take some time for Appeears to process your request - anything from a few minutes to a few hours depending on how busy they are. You can check your progress by:

1. Going to the [Appeears webpage](https://appeears.earthdatacloud.nasa.gov/)  
2. Clicking the `Explore` tab  
3. Logging in with your Earthdata account

In [22]:
### make a home for the data
project = earthpy.project.Project(dirname = str(ndvi_dir))  ### doesn't want a pathlib object, so convert to string
project.project_dir

WindowsPath('c:/users/nyca4176/repos/earth-data-portfolio/coding-assignments/03_vegetation/data/06_ndvi')

In [ ]:
### make descriptive download key for the data you'll download
download_key = "RMNP NDVI"

### initialize AppeearsDownloader for MODIS NDVI data
ndvi_downloader = eaapp.AppeearsDownloader(

    ### add download key
    download_key = download_key,

    ### define folde where data will download to
    project = project,

    ### specify remote sensing product
    product = 'MOD13Q1.061',

    ### and specific layer
    layer = '_250m_16_days_NDVI',

    ### start date (change this to July)
    start_date = "07-01",

    ### end data (change this to July as well)
    end_date = "07-31",

    recurring = True,

    ### set year range
    year_range = [2022, 2025],

    ### set polygon for area of interest
    polygon = romo_gdf
)

### download the prepared data (can be slow)
ndvi_downloader.download_files(cache=True)

You can now pick up the data wrangling process from 03-vegetation-wrangle.ipynb starting at Step 3A.